# Data Audit — Diabetes 130-US Hospitals Dataset

This notebook audits the raw dataset before any cleaning or modelling. The goal is to understand structure, identify data quality issues, and produce a written cleaning plan that drives `02_data_cleaning.ipynb`.

---
## 1. Project Context

### 1.1 Problem Statement

Hospital readmissions within 30 days are a major cost driver for healthcare systems and often signal gaps in post-discharge care. This project uses the **Diabetes 130-US Hospitals** dataset — drawn from 130 US hospitals between 1999–2008 — to predict which diabetic patients are likely to be readmitted within 30 days of discharge.

By identifying high-risk patients at the point of discharge, hospitals can prioritise targeted interventions: follow-up calls, medication reviews, and structured care plans.

### 1.2 Business Objective

Deliver a risk-scoring tool that flags patients at high risk of 30-day readmission, enabling clinical teams to intervene before discharge and reduce avoidable returns.

### 1.3 Analytical Objective

Build a **binary classification model** predicting:
- `1` = readmitted within 30 days (`<30`)
- `0` = not readmitted within 30 days (`>30` or `NO`)

### 1.4 Success Criteria

Model performance will be evaluated using **ROC-AUC** and **Recall** as primary metrics, with Precision as secondary. In a clinical risk-detection setting, missing a high-risk patient (false negative) is more costly than a false alarm, so recall is prioritised alongside overall discrimination ability.

### 1.5 Scope

**In scope:** data cleaning, exploratory analysis, feature engineering, model building, and performance evaluation.

**Out of scope:** real-time clinical deployment, integration with external patient records or live hospital systems.

---
## 2. Load Data

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_loader import load_raw_data

data = load_raw_data()
print(f"Shape: {data.shape}")
data.head()

Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


---
## 3. Dataset Overview

### 3.1 Dimensions

The dataset contains **101,766 hospital encounters** across **50 columns**. Each row represents a single inpatient encounter — not a unique patient, as some patients appear multiple times.

In [2]:
print(f"Rows:    {data.shape[0]:,}")
print(f"Columns: {data.shape[1]}")
print(f"\nColumn names:")
for i, col in enumerate(data.columns, 1):
    print(f"  {i:2}. {col}")

Rows:    101,766
Columns: 50

Column names:
   1. encounter_id
   2. patient_nbr
   3. race
   4. gender
   5. age
   6. weight
   7. admission_type_id
   8. discharge_disposition_id
   9. admission_source_id
  10. time_in_hospital
  11. payer_code
  12. medical_specialty
  13. num_lab_procedures
  14. num_procedures
  15. num_medications
  16. number_outpatient
  17. number_emergency
  18. number_inpatient
  19. diag_1
  20. diag_2
  21. diag_3
  22. number_diagnoses
  23. max_glu_serum
  24. A1Cresult
  25. metformin
  26. repaglinide
  27. nateglinide
  28. chlorpropamide
  29. glimepiride
  30. acetohexamide
  31. glipizide
  32. glyburide
  33. tolbutamide
  34. pioglitazone
  35. rosiglitazone
  36. acarbose
  37. miglitol
  38. troglitazone
  39. tolazamide
  40. examide
  41. citoglipton
  42. insulin
  43. glyburide-metformin
  44. glipizide-metformin
  45. glimepiride-pioglitazone
  46. metformin-rosiglitazone
  47. metformin-pioglitazone
  48. change
  49. diabetesMed
  50. 

### 3.2 Column Groups

The 50 columns fall into five logical groups:

| Group | Columns | Description |
|---|---|---|
| **Identifiers** | `encounter_id`, `patient_nbr` | Row and patient keys — not predictive features |
| **Demographics** | `race`, `gender`, `age`, `weight` | Patient characteristics |
| **Admission / Discharge** | `admission_type_id`, `discharge_disposition_id`, `admission_source_id`, `time_in_hospital` | Hospital encounter metadata |
| **Clinical** | `payer_code`, `medical_specialty`, `num_lab_procedures`, `num_procedures`, `num_medications`, `number_outpatient`, `number_emergency`, `number_inpatient`, `diag_1–3`, `number_diagnoses`, `max_glu_serum`, `A1Cresult` | Lab results, diagnoses, service utilisation |
| **Medications** | 23 drug columns + `change`, `diabetesMed` | Individual diabetes medication dosage changes |
| **Target** | `readmitted` | Outcome variable: `<30`, `>30`, `NO` |

In [3]:
data.dtypes.value_counts()

dtype
object    36
int64     14
dtype: int64

---
## 4. Missing Value Analysis

### 4.1 Genuine NaN

Pandas `isnull()` will catch true `NaN` / `None` values.

In [4]:
null_counts = data.isnull().sum()
null_pct = (null_counts / len(data) * 100).round(1)

null_summary = pd.DataFrame({'missing_count': null_counts, 'missing_pct': null_pct})
null_summary = null_summary[null_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)

print("Columns with genuine NaN:")
print(null_summary)

Columns with genuine NaN:
               missing_count  missing_pct
max_glu_serum          96420         94.7
A1Cresult              84748         83.3
weight                 98569         96.9


### 4.2 '?' Placeholder Strings

The dataset also encodes missing values as the string `'?'`. These are invisible to `isnull()` and must be identified separately.

In [5]:
# Check all object columns for '?' placeholders
obj_cols = data.select_dtypes(include='object').columns
q_counts = {col: (data[col] == '?').sum() for col in obj_cols}
q_counts = {k: v for k, v in q_counts.items() if v > 0}

print("Columns containing '?' placeholders:")
for col, count in sorted(q_counts.items(), key=lambda x: -x[1]):
    pct = count / len(data) * 100
    print(f"  {col:<25} {count:>6,}  ({pct:.1f}%)")

Columns containing '?' placeholders:
  medical_specialty         49949  (49.1%)
  payer_code                40256  (39.6%)
  race                       2273  ( 2.2%)
  diag_3                     1423  ( 1.4%)
  diag_2                      358  ( 0.4%)
  diag_1                       21  ( 0.2%)


> **Note:** `max_glu_serum` and `A1Cresult` have both genuine NaN (where the test was never run) and are already captured above. The `?` placeholder appears in demographic and clinical text fields.

---
## 5. Target Variable Analysis

The target column `readmitted` has three categories. For binary classification we will collapse `>30` and `NO` into a single negative class.

In [6]:
target_counts = data['readmitted'].value_counts()
target_pct    = (target_counts / len(data) * 100).round(1)

print("readmitted value distribution:")
for val, cnt in target_counts.items():
    print(f"  {val:<5}  {cnt:>6,}  ({target_pct[val]}%)")

print(f"\n→ Positive class (<30): {target_counts['<30']:,} ({target_pct['<30']}%)")
print(f"→ Negative class (>30 + NO): {target_counts['>30'] + target_counts['NO']:,} ({100 - target_pct['<30']:.1f}%)")
print("\n⚠ Dataset is class-imbalanced. Recall and ROC-AUC are the appropriate primary metrics.")

readmitted value distribution:
  NO     54864  (53.9%)
  >30    35545  (34.9%)
  <30    11357  (11.2%)

→ Positive class (<30): 11,357 (11.2%)
→ Negative class (>30 + NO): 90,409 (88.8%)

⚠ Dataset is class-imbalanced. Recall and ROC-AUC are the appropriate primary metrics.


---
## 6. Key Column Inspection

### 6.1 Identifier Columns

`encounter_id` and `patient_nbr` are unique-per-row and patient identifiers respectively. They carry no predictive information and would cause data leakage if included in a model.

In [7]:
print(f"encounter_id — unique values: {data['encounter_id'].nunique():,} / {len(data):,} rows")
print(f"patient_nbr  — unique values: {data['patient_nbr'].nunique():,} / {len(data):,} rows")
print(f"\n→ {data['patient_nbr'].nunique():,} unique patients across {len(data):,} encounters")
print(f"→ Average encounters per patient: {len(data) / data['patient_nbr'].nunique():.2f}")

encounter_id — unique values: 101,766 / 101,766 rows
patient_nbr  — unique values: 71,518 / 101,766 rows

→ 71,518 unique patients across 101,766 encounters
→ Average encounters per patient: 1.42


### 6.2 Weight Column

`weight` is 96.9% missing. Imputing this volume of data would introduce more noise than signal, so it will be dropped.

In [8]:
weight_missing = (data['weight'] == '?').sum()
print(f"weight — '?' values: {weight_missing:,} ({weight_missing/len(data)*100:.1f}%)")
print(f"weight — non-missing: {len(data) - weight_missing:,} ({(len(data)-weight_missing)/len(data)*100:.1f}%)")

weight — '?' values: 98,569 (96.9%)
weight — non-missing: 3,197 (3.1%)


### 6.3 Age Column

`age` is stored as decade-bracket strings (e.g. `[60-70)`). It will be converted to numeric midpoints for modelling.

In [9]:
print("age value distribution:")
print(data['age'].value_counts().sort_index())

age value distribution:
age
[0-10)        161
[10-20)       691
[20-30)      1657
[30-40)      3775
[40-50)      9685
[50-60)     17256
[60-70)     22483
[70-80)     26068
[80-90)     17197
[90-100)     2793
Name: count, dtype: int64


### 6.4 Coded ID Columns

`admission_type_id`, `discharge_disposition_id`, and `admission_source_id` store categorical information as numeric codes. These will be mapped to descriptive labels using the `IDS_mapping.csv` reference file.

In [10]:
id_cols = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in id_cols:
    print(f"{col}: {sorted(data[col].unique())}")

admission_type_id: [1, 2, 3, 4, 5, 6, 7, 8]
discharge_disposition_id: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 23, 24, 25, 27, 28, 29]
admission_source_id: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 17, 20, 22, 25]


### 6.5 Lab Test Columns

`max_glu_serum` (maximum glucose serum result) and `A1Cresult` (HbA1c test result) are primarily missing because the tests were not ordered for most encounters. The absence itself is informative — patients with no test recorded differ clinically from those who were tested.

In [11]:
for col in ['max_glu_serum', 'A1Cresult']:
    print(f"\n{col}:")
    print(data[col].value_counts(dropna=False))


max_glu_serum:
max_glu_serum
NaN     96420
None     4523
Norm     3326
>200     1385
>300      112
Name: count, dtype: int64

A1Cresult:
A1Cresult
NaN     84748
None    14737
Norm     7644
>7       4893
>8         30
Name: count, dtype: int64


---
## 7. Cleaning Plan

Based on the audit findings above, the following actions will be taken in `02_data_cleaning.ipynb`:

| # | Action | Columns | Rationale |
|---|---|---|---|
| 1 | **Drop** | `weight`, `encounter_id`, `patient_nbr` | 97% missing; no predictive value / data leakage risk |
| 2 | **Replace `?` with `'Unknown'`** | `race`, `payer_code`, `medical_specialty`, `diag_1`, `diag_2`, `diag_3` | Preserve the fact that the value was not recorded |
| 3 | **Fill `NaN` with `'Unknown'`** | `max_glu_serum`, `A1Cresult` | Test not performed — absence is its own category |
| 4 | **Binarise target** | `readmitted` → `readmitted_binary` | `1` if `<30`, else `0`; enables binary classification |
| 5 | **Convert age to numeric** | `age` → `age_numeric` | Midpoint of each bracket (e.g. `[60-70)` → `65`) |
| 6 | **Map coded IDs to labels** | `admission_type_id`, `discharge_disposition_id`, `admission_source_id` | Convert integer codes to readable categories via `IDS_mapping.csv` |
| 7 | **Drop original columns** | `age`, `readmitted`, the three `_id` columns | Superseded by their cleaned replacements |